# ⚡ 140 Mya Cretaceous Extinct Ancestor (10.0 ns All-Atom Explicit Solvent MD)
### Paleogenomic Neofunctionalization & Physical Gate Stability
**Author:** Bruno ('Blu') Stein Barbosa Menechino ([PMID: 38049805](https://pubmed.ncbi.nlm.nih.gov/38049805/))  
**Engine:** OpenMM with Native NVIDIA CUDA GPU Platform  
**Hardware:** Google Colab T4 GPU  
**Protocol:** 10.0 ns Production MD in explicit TIP3P solvent (0.15 M NaCl) with 4.0 fs HMR timestep, live speed & ETA reporting, backbone RMSD analysis, and automated ZIP packaging.

---
### 🚀 Quick Start:
1. Ensure runtime is GPU: **Runtime -> Change runtime type -> T4 GPU**.
2. Click **Runtime -> Run all** (`Ctrl+F9`).
*The 140 Mya ancestral coordinates are embedded directly in this notebook!*

## Step 1: Install OpenMM (CUDA 12) & MDTraj

In [ ]:
!pip install -q "openmm[cuda12]" openmm-cuda-12 mdtraj matplotlib seaborn pandas scipy

import openmm as mm
avail = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print(f"Available Platforms: {avail}")
if "CUDA" in avail:
    print("✅ Active GPU Engine: CUDA (NVIDIA T4 detected)")
else:
    print(f"⚠️ Falling back to: {avail[0]}")
!nvidia-smi


## Step 2: Unpack Embedded 140 Mya Extinct Ancestor Coordinates

In [ ]:
import os, sys, base64, gzip

BASE_DIR = os.path.abspath("ancestral_md")
INP_DIR = os.path.join(BASE_DIR, "inputs")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(INP_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

pdb_b64 = "..."
raw = gzip.decompress(base64.b64decode(pdb_b64))
pdb_path = os.path.join(INP_DIR, "5_Anc0_140Mya_Extinct_Ancestor_SLU_holo.pdb")
with open(pdb_path, "wb") as f:
    f.write(raw)
print(f"✅ Extracted: {pdb_path} ({len(raw)} bytes)")


## Step 3: Run 10.0 ns Explicit-Solvent GPU MD (4.0 fs HMR + Live Logging)

In [ ]:
import os, sys, io, time
import numpy as np
import mdtraj as md
import openmm as mm
from openmm import app, unit

BASE_DIR = os.path.abspath("ancestral_md")
INP_DIR = os.path.join(BASE_DIR, "inputs")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
anc_dir = os.path.join(OUT_DIR, "5_Anc0_140Mya_Extinct_Ancestor")
os.makedirs(anc_dir, exist_ok=True)

dcd_path = os.path.join(anc_dir, "prod.dcd")
top_path = os.path.join(anc_dir, "solvated.pdb")
log_path = os.path.join(anc_dir, "sim_progress.csv")

if os.path.exists(dcd_path) and os.path.getsize(dcd_path) > 500000:
    print(f"✅ Ancestral simulation already complete ({dcd_path}). Skipping!")
else:
    print("=" * 70, flush=True)
    print("🚀 Launching 10.0 ns All-Atom GPU MD on 140 Mya Cretaceous Ancestor", flush=True)
    print("=" * 70, flush=True)

    pdb_in = os.path.join(INP_DIR, "5_Anc0_140Mya_Extinct_Ancestor_SLU_holo.pdb")

    # 1. Ingest protein coordinates (clean protein ATOM lines to isolate APO tertiary fold)
    with open(pdb_in, "r") as fh:
        clean_lines = [l for l in fh if l.startswith(("ATOM", "TER", "END"))]
    pdb = app.PDBFile(io.StringIO("".join(clean_lines)))

    # 2. Build AMBER99SB-ILDN + TIP3P Force Field
    print("  [1/5] Adding hydrogens and solvating box (0.15 M NaCl)...", flush=True)
    forcefield = app.ForceField("amber99sbildn.xml", "tip3p.xml")
    modeller = app.Modeller(pdb.topology, pdb.positions)
    modeller.addHydrogens(forcefield)
    modeller.addSolvent(forcefield, model="tip3p", padding=1.0*unit.nanometer, ionicStrength=0.15*unit.molar)

    # 3. Create System with HMR (dt = 4.0 fs enabled by hydrogenMass = 3.0 amu)
    print("  [2/5] Parameterizing system with HMR (dt = 4.0 fs)...", flush=True)
    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=app.PME,
        nonbondedCutoff=1.0*unit.nanometer,
        constraints=app.AllBonds,
        hydrogenMass=3.0*unit.amu
    )
    system.addForce(mm.MonteCarloBarostat(1.0*unit.bar, 300.0*unit.kelvin, 25))

    integrator = mm.LangevinMiddleIntegrator(300.0*unit.kelvin, 1.0/unit.picosecond, 0.004*unit.picoseconds)
    platform = mm.Platform.getPlatformByName("CUDA")
    properties = {"Precision": "mixed", "DeviceIndex": "0"}

    simulation = app.Simulation(modeller.topology, system, integrator, platform, properties)
    simulation.context.setPositions(modeller.positions)

    # 4. Energy Minimization (5,000 steps)
    print("  [3/5] Minimizing potential energy...", flush=True)
    simulation.minimizeEnergy(maxIterations=5000)

    # Save minimized solvated reference topology for mdtraj
    min_state = simulation.context.getState(getPositions=True)
    with open(top_path, "w") as fh:
        app.PDBFile.writeFile(simulation.topology, min_state.getPositions(), fh)
    print(f"  Solvated system: {simulation.topology.getNumAtoms()} atoms ({simulation.topology.getNumResidues()} residues)", flush=True)

    # 5. Fast NPT Equilibration (100 ps = 25,000 steps)
    print("  [4/5] Equilibrating solvent & protein at 300 K / 1.0 bar (100 ps)...", flush=True)
    simulation.step(25000)

    # 6. Production MD on NVIDIA GPU (10.0 ns in 100 chunks with LIVE FLUSHED LOGGING)
    total_steps = 2500000
    chunk_steps = 25000 # 0.1 ns per chunk (~40-50s on T4 GPU)
    n_chunks = total_steps // chunk_steps
    simulation.reporters.append(app.DCDReporter(dcd_path, 5000)) # Save frame every 20 ps (500 frames)

    print(f"  [5/5] Running 10.0 ns Production MD on NVIDIA GPU ({n_chunks} live checkpoints)...", flush=True)
    print("  " + "-" * 68, flush=True)

    with open(log_path, "w") as log_fh:
        log_fh.write("step,time_ns,speed_ns_per_day,epot_kj\n")

    t_start = time.time()
    for chunk in range(1, n_chunks + 1):
        t_c0 = time.time()
        simulation.step(chunk_steps)
        t_c1 = time.time()

        step_done = chunk * chunk_steps
        pct = (step_done / total_steps) * 100.0
        sim_ns = step_done * 0.004

        chunk_dur = t_c1 - t_c0
        chunk_speed = (0.1 / chunk_dur) * 86400.0 if chunk_dur > 0 else 0.0
        tot_elapsed = t_c1 - t_start
        avg_speed = (sim_ns / tot_elapsed) * 86400.0
        rem_sec = ((10.0 - sim_ns) / (avg_speed / 86400.0)) if avg_speed > 0 else 0
        rem_m = int(rem_sec // 60)
        rem_s = int(rem_sec % 60)

        state = simulation.context.getState(getEnergy=True)
        epot = state.getPotentialEnergy().value_in_unit(unit.kilojoule / unit.mole)
        print(f"  ⚡ [{pct:5.1f}%] {sim_ns:5.2f} / 10.00 ns | Speed: {chunk_speed:5.1f} ns/day (Avg: {avg_speed:5.1f}) | ETA: {rem_m:02d}m {rem_s:02d}s | Epot: {epot:,.0f} kJ/mol", flush=True)

        with open(log_path, "a") as log_fh:
            log_fh.write(f"{step_done},{sim_ns:.4f},{avg_speed:.2f},{epot:.1f}\n")

    dur = (time.time() - t_start) / 3600.0
    print(f"\n✅ Finished 10.0 ns in {dur:.2f} hours for 140 Mya Extinct Ancestor!", flush=True)


## Step 4: Trajectory Backbone Stability & Catalytic Gate Loop Analysis

In [ ]:
import os
import matplotlib.pyplot as plt
import mdtraj as md
import numpy as np

BASE_DIR = os.path.abspath("ancestral_md")
anc_dir = os.path.join(BASE_DIR, "outputs", "5_Anc0_140Mya_Extinct_Ancestor")
top_file = os.path.join(anc_dir, "solvated.pdb")
dcd_file = os.path.join(anc_dir, "prod.dcd")

if os.path.exists(top_file) and os.path.exists(dcd_file):
    traj = md.load(dcd_file, top=top_file)
    bb_atoms = traj.topology.select("backbone")
    rmsd = md.rmsd(traj, traj, frame=0, atom_indices=bb_atoms) * 10.0 # nm to Angstroms
    time_ns = traj.time / 1000.0

    loop_atoms = traj.topology.select("resid 343 to 350 and backbone")
    if len(loop_atoms) > 0:
        loop_rmsd = md.rmsd(traj, traj, frame=0, atom_indices=loop_atoms) * 10.0
    else:
        loop_rmsd = np.zeros_like(rmsd)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=180)

    # Panel 1: Full Backbone RMSD
    ax1.plot(time_ns, rmsd, color="#d97706", lw=2.0, label=f"140 Mya Ancestor Backbone (Final={rmsd[-1]:.2f} Å)")
    ax1.set_xlabel("Simulation Time (ns)", fontweight="bold")
    ax1.set_ylabel("Backbone RMSD (Å)", fontweight="bold")
    ax1.set_title("140 Mya Cretaceous Extinct Ancestor: 10 ns Stability", fontweight="bold")
    ax1.grid(alpha=0.3)
    ax1.legend(loc="lower right")

    # Panel 2: Catalytic Gate Loop (Residues 343-350)
    ax2.plot(time_ns, loop_rmsd, color="#dc2626", lw=2.0, label=f"Gate Loop 343-350 (Final={loop_rmsd[-1]:.2f} Å)")
    ax2.set_xlabel("Simulation Time (ns)", fontweight="bold")
    ax2.set_ylabel("Loop RMSD (Å)", fontweight="bold")
    ax2.set_title("Catalytic Gate Loop Flexibility (L347 Plug)", fontweight="bold")
    ax2.grid(alpha=0.3)
    ax2.legend(loc="lower right")

    plt.tight_layout()
    analysis_img = os.path.join(anc_dir, "ancestral_10ns_analysis.png")
    plt.savefig(analysis_img, dpi=200)
    plt.show()
    print(f"✅ Trajectory analysis saved to: {analysis_img}")


## Step 5: Package Trajectory and Artifacts into Zip for Direct Download

In [ ]:
import os, zipfile
from google.colab import files

BASE_DIR = os.path.abspath("ancestral_md")
anc_dir = os.path.join(BASE_DIR, "outputs", "5_Anc0_140Mya_Extinct_Ancestor")
zip_path = os.path.join(BASE_DIR, "outputs", "140Mya_Ancestral_MD_10ns.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, filenames in os.walk(anc_dir):
        for f in filenames:
            fp = os.path.join(root, f)
            arcname = os.path.relpath(fp, anc_dir)
            zipf.write(fp, arcname)

print(f"✅ Created ZIP Archive: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)")
print("🚀 Triggering browser download...")
files.download(zip_path)
